In [ ]:
# Install the package (uncomment if not already installed)
#!pip install git+https://github.com/sebastianprobst/resonator_tools.git

In [ ]:
import numpy as np
import pandas as pd
from resonator_tools import circuit

In [ ]:
# List of file base names (without extension) to process
file_names = [
    'TWPA_on_S21_4.30-4.50GHz_-35.0dBm_20250604_182125',
    'TWPA_on_S21_4.42-4.47GHz_-25.0dBm_20250605_205539',
    'TWPA_on_S21_4.91-4.93GHz_-25.0dBm_20250605_205559',
    'TWPA_on_S21_5.20-5.35GHz_-35.0dBm_20250605_200813',
    'TWPA_on_S21_5.55-5.65GHz_-25.0dBm_20250605_205646',
    'TWPA_on_S21_6.60-6.90GHz_-35.0dBm_20250605_192037',
    'TWPA_on_S21_6.95-7.05GHz_-35.0dBm_20250605_193000',
    'TWPA_on_S21_7.50-8.00GHz_-35.0dBm_20250605_201822',
    'TWPA_on_S21_7.65-7.75GHz_-35.0dBm_20250605_202005',
]

data_folder = 'Data'

In [ ]:
def fit_q_factors(csv_path):
    """Load an S21 sweep CSV and return the notch-port fit results dict."""
    df = pd.read_csv(csv_path)

    # Convert units
    freq = df['Frequency (GHz)'].values * 1e9
    mag_lin = 10**(df['Magnitude (dB)'].values / 20)
    phase_rad = np.deg2rad(df['Phase (deg)'].values)
    s21_complex = mag_lin * np.exp(1j * phase_rad)

    fit = circuit.notch_port()
    fit.add_data(freq, s21_complex)

    delay, a, alpha, fr, Ql, A2, frcal = fit.do_calibration(fit.f_data, fit.z_data_raw)
    fit.z_data = fit.do_normalization(fit.f_data, fit.z_data_raw, delay, a, alpha, A2, frcal)
    fit.fitresults = fit.circlefit(fit.f_data, fit.z_data, fr, Ql)

    return fit.fitresults

In [ ]:
results = []

for name in file_names:
    csv_path = f'{data_folder}/{name}.csv'
    try:
        fitresults = fit_q_factors(csv_path)
        results.append({
            'file': name,
            'fr_Hz': fitresults['fr'],
            'Ql': fitresults['Ql'],
            'Qc_dia_corr': fitresults['Qc_dia_corr'],
            'Qi_dia_corr': fitresults['Qi_dia_corr'],
        })
        print(f"{name}\n"
              f"  fr = {fitresults['fr']/1e9:.4f} GHz | Ql = {fitresults['Ql']:.2e} | "
              f"Qc = {fitresults['Qc_dia_corr']:.2e} | Qi = {fitresults['Qi_dia_corr']:.2e}\n")
    except Exception as e:
        print(f"Failed to fit {name}: {e}\n")
        results.append({
            'file': name,
            'fr_Hz': np.nan,
            'Ql': np.nan,
            'Qc_dia_corr': np.nan,
            'Qi_dia_corr': np.nan,
        })

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv(f'{data_folder}/Q_factor_results.csv', index=False)
results_df